In [5]:
import torch
import transformers

In [3]:
model = transformers.AutoModel.from_pretrained('aehrc/uniformer_base_tl_384')

The repository for aehrc/uniformer_base_tl_384 contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/aehrc/uniformer_base_tl_384.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
The repository for aehrc/uniformer_base_tl_384 contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/aehrc/uniformer_base_tl_384.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


In [4]:
model

UniFormerModel(
  (uniformer): UniFormer(
    (patch_embed1): PatchEmbed(
      (proj): Conv2d(3, 64, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (patch_embed2): PatchEmbed(
      (proj): Conv2d(64, 128, kernel_size=(2, 2), stride=(2, 2))
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (patch_embed3): PatchEmbed(
      (proj): Conv2d(128, 320, kernel_size=(2, 2), stride=(2, 2))
      (norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
    )
    (patch_embed4): PatchEmbed(
      (proj): Conv2d(320, 512, kernel_size=(2, 2), stride=(2, 2))
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks1): ModuleList(
      (0): CBlock(
        (pos_embed): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64)
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_

In [5]:
fake_image_batch = torch.randn(1, 3, 384, 384)  # (batch, channels, time, height, width)

In [6]:
output = model(fake_image_batch)

In [7]:
output.last_hidden_state.shape

torch.Size([1, 144, 512])

In [8]:
output.last_hidden_state.max(1)[0].shape

torch.Size([1, 512])

In [9]:
fake_image_batch = torch.randn(1, 3, 448, 448)  # (batch, channels, time, height, width)
output = model(fake_image_batch)

AssertionError: Input image size (448*448) doesn't match model (384*384).

In [1]:
from medvqa.models.vision.visual_modules import create_huggingface_cxrmate_rrg24_uniformer_feature_extractor

In [2]:
model = create_huggingface_cxrmate_rrg24_uniformer_feature_extractor('aehrc/cxrmate-rrg24', None)

In [3]:
model

UniFormer(
  (patch_embed1): PatchEmbed(
    (proj): Conv2d(3, 64, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (patch_embed2): PatchEmbed(
    (proj): Conv2d(64, 128, kernel_size=(2, 2), stride=(2, 2))
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (patch_embed3): PatchEmbed(
    (proj): Conv2d(128, 320, kernel_size=(2, 2), stride=(2, 2))
    (norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
  )
  (patch_embed4): PatchEmbed(
    (proj): Conv2d(320, 512, kernel_size=(2, 2), stride=(2, 2))
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks1): ModuleList(
    (0): CBlock(
      (pos_embed): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64)
      (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 

In [6]:
fake_image_batch = torch.randn(1, 3, 384, 384)  # (batch, channels, time, height, width)

In [8]:
out = model(fake_image_batch)

In [11]:
out.shape

torch.Size([1, 512, 12, 12])

In [12]:
fake_image_batch = torch.randn(1, 3, 448, 448)  # (batch, channels, time, height, width)

In [13]:
out = model(fake_image_batch)

AssertionError: Input image size (448*448) doesn't match model (384*384).

In [15]:
import torch
from transformers import AutoModel
from timm.models.layers import to_2tuple

# 1. Load the model as you did before
model = create_huggingface_cxrmate_rrg24_uniformer_feature_extractor('aehrc/cxrmate-rrg24', None)

# 2. Define your new image resolution
new_image_size = 448
new_image_size_tuple = to_2tuple(new_image_size)

# 3. Update the image_size attribute for each patch embedding layer
# The image size for each subsequent layer is the feature map size from the previous one.
model.patch_embed1.image_size = new_image_size_tuple
ps1 = model.patch_embed1.patch_size
# After patch_embed1, size is 448/4 = 112
h, w = new_image_size_tuple[0] // ps1[0], new_image_size_tuple[1] // ps1[1]
model.patch_embed2.image_size = (h, w)
ps2 = model.patch_embed2.patch_size
# After patch_embed2, size is 112/2 = 56
h, w = h // ps2[0], w // ps2[1]
model.patch_embed3.image_size = (h, w)
ps3 = model.patch_embed3.patch_size
# After patch_embed3, size is 56/2 = 28
h, w = h // ps3[0], w // ps3[1]
model.patch_embed4.image_size = (h, w)

print(f"Updated patch_embed1 image_size to: {model.patch_embed1.image_size}")
print(f"Updated patch_embed2 image_size to: {model.patch_embed2.image_size}")
print(f"Updated patch_embed3 image_size to: {model.patch_embed3.image_size}")
print(f"Updated patch_embed4 image_size to: {model.patch_embed4.image_size}")


# 4. Now the model will accept the new resolution without error
fake_image_batch = torch.randn(1, 3, new_image_size, new_image_size)
out = model(fake_image_batch)

print(f"\nSuccessfully processed a {new_image_size}x{new_image_size} image.")
print(f"Output shape: {out.shape}")

# The output spatial dimension will be 14x14 (448 / (4*2*2*2))
# Expected: torch.Size([1, 512, 14, 14])
assert out.shape == (1, 512, 14, 14)

Updated patch_embed1 image_size to: (448, 448)
Updated patch_embed2 image_size to: (112, 112)
Updated patch_embed3 image_size to: (56, 56)
Updated patch_embed4 image_size to: (28, 28)

Successfully processed a 448x448 image.
Output shape: torch.Size([1, 512, 14, 14])
